In [ ]:
# Reset conflicting packages
!pip uninstall -y numpy spacy thinc

# Install compatible versions
!pip install -q numpy==1.26.4
!pip install -q spacy==3.7.4
!pip install -q thinc==8.2.3

# Install other project dependencies
!pip install -q langchain langchain-community
!pip install -q faiss-cpu
!pip install -q pypdf2 python-docx python-pptx
!pip install -q nltk streamlit pandas

# Download spaCy model
!python -m spacy download en_core_web_sm

Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4
Found existing installation: spacy 3.7.2
Uninstalling spacy-3.7.2:
  Successfully uninstalled spacy-3.7.2
Found existing installation: thinc 8.2.5
Uninstalling thinc-8.2.5:
  Successfully uninstalled thinc-8.2.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastai 2.8.7 requires spacy<4, which is not installed.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible

In [ ]:
import os
import sys
from pathlib import Path
import logging
from typing import List, Dict, Any, Optional

import PyPDF2
from docx import Document
from pptx import Presentation

import spacy
import nltk

from dataclasses import dataclass
from enum import Enum
import hashlib
import json
from datetime import datetime

nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
class DocumentType(str, Enum):
    """Document type classification"""
    TEXTBOOK = "textbook"
    LECTURE_NOTES = "lecture_notes"
    QUESTION_PAPER = "question_paper"
    LAB_MANUAL = "lab_manual"
    SLIDES = "slides"
    OTHER = "other"

@dataclass
class ProcessedDocument:
    """Data class for processed document"""
    content: str
    metadata: Dict[str, Any]
    document_type: DocumentType
    chunks: List[str] = None

In [ ]:
class PDFLoader:
    """PDF document loader with text extraction"""

    def __init__(self):
        self.supported_extensions = ['.pdf']
        logger.info("PDFLoader initialized")

    def load(self, file_path: str) -> List[ProcessedDocument]:
        """
        Load and extract text from PDF file
        """
        documents = []

        try:
            with open(file_path, 'rb') as file:
                pdf_reader = PyPDF2.PdfReader(file)
                for page_num, page in enumerate(pdf_reader.pages):
                    text = page.extract_text()

                    if text and text.strip():
                        # Create metadata
                        metadata = {
                            'source': file_path,
                            'page': page_num + 1,
                            'total_pages': len(pdf_reader.pages),
                            'file_type': 'pdf',
                            'file_name': os.path.basename(file_path),
                            'file_size': os.path.getsize(file_path),
                            'extraction_date': datetime.now().isoformat()
                        }
                        doc = ProcessedDocument(
                            content=text.strip(),
                            metadata=metadata,
                            document_type=self._classify_document(text)
                        )
                        documents.append(doc)

            logger.info(f"Successfully loaded {len(documents)} pages from {file_path}")
            return documents

        except Exception as e:
            logger.error(f"Error loading PDF {file_path}: {str(e)}")
            raise

    def _classify_document(self, text: str) -> DocumentType:
        """Basic document classification based on content"""
        text_lower = text.lower()
        # Simple keyword-based classification
        if any(word in text_lower for word in ['textbook', 'chapter', 'exercise', 'review questions']):
            return DocumentType.TEXTBOOK
        elif any(word in text_lower for word in ['lecture', 'notes', 'professor', 'class']):
            return DocumentType.LECTURE_NOTES
        elif any(word in text_lower for word in ['question paper', 'exam', 'marks', 'section a', 'section b']):
            return DocumentType.QUESTION_PAPER
        elif any(word in text_lower for word in ['lab', 'experiment', 'apparatus', 'procedure']):
            return DocumentType.LAB_MANUAL
        elif any(word in text_lower for word in ['slide', 'presentation', 'overview']):
            return DocumentType.SLIDES
        else:
            return DocumentType.OTHER

In [ ]:
class DOCXLoader:
    """DOCX document loader"""

    def __init__(self):
        self.supported_extensions = ['.docx']
        logger.info("DOCXLoader initialized")

    def load(self, file_path: str) -> List[ProcessedDocument]:
        """
        Load and extract text from DOCX file
        """
        documents = []

        try:
            doc = Document(file_path)
            full_text = []

            # Extract paragraphs
            for para in doc.paragraphs:
                if para.text.strip():
                    full_text.append(para.text)
                    # Extract tables
            for table in doc.tables:
                for row in table.rows:
                    row_text = [cell.text for cell in row.cells if cell.text.strip()]
                    if row_text:
                        full_text.append(' | '.join(row_text))

            content = '\n'.join(full_text)

            # Create metadata
            metadata = {
                'source': file_path,
                'file_type': 'docx',
                'file_name': os.path.basename(file_path),
                'file_size': os.path.getsize(file_path),
                'paragraph_count': len(doc.paragraphs),
                'table_count': len(doc.tables),
                'extraction_date': datetime.now().isoformat()
                 }

            # Create processed document
            doc_obj = ProcessedDocument(
                content=content,
                metadata=metadata,
                document_type=self._classify_document(content)
            )
            documents.append(doc_obj)

            logger.info(f"Successfully loaded DOCX: {file_path}")
            return documents

        except Exception as e:
            logger.error(f"Error loading DOCX {file_path}: {str(e)}")
            raise

    def _classify_document(self, text: str) -> DocumentType:
        """Basic document classification"""
        text_lower = text.lower()
        if any(word in text_lower for word in ['textbook', 'chapter', 'exercise']):
            return DocumentType.TEXTBOOK
        elif any(word in text_lower for word in ['lecture', 'notes']):
            return DocumentType.LECTURE_NOTES
        elif any(word in text_lower for word in ['question', 'exam', 'marks']):
            return DocumentType.QUESTION_PAPER
        elif any(word in text_lower for word in ['lab', 'experiment']):
            return DocumentType.LAB_MANUAL
        else:
            return DocumentType.OTHER

In [ ]:
class PPTXLoader:
    """PowerPoint document loader"""

    def __init__(self):
        self.supported_extensions = ['.pptx']
        logger.info("PPTXLoader initialized")

    def load(self, file_path: str) -> List[ProcessedDocument]:
        """
        Load and extract text from PPTX file
        """
        documents = []

        try:
            prs = Presentation(file_path)

            for slide_num, slide in enumerate(prs.slides):
                slide_text = []
                for shape in slide.shapes:
                    if hasattr(shape, "text"):
                        if shape.text.strip():
                            slide_text.append(shape.text)
                    elif hasattr(shape, "table"):
                        # Extract table content
                        for row in shape.table.rows:
                            row_text = [cell.text for cell in row.cells if cell.text.strip()]
                            if row_text:
                                slide_text.append(' | '.join(row_text))

                if slide_text:
                    content = '\n'.join(slide_text)

                    # Create metadata
                    metadata = {
                        'source': file_path,
                        'slide': slide_num + 1,
                                                'total_slides': len(prs.slides),
                        'file_type': 'pptx',
                        'file_name': os.path.basename(file_path),
                        'file_size': os.path.getsize(file_path),
                        'extraction_date': datetime.now().isoformat()
                    }

                    # Create processed document
                    doc = ProcessedDocument(
                        content=content,
                        metadata=metadata,
                        document_type=self._classify_document(content)
                    )
                    documents.append(doc)

            logger.info(f"Successfully loaded {len(documents)} slides from {file_path}")
            return documents

        except Exception as e:
            logger.error(f"Error loading PPTX {file_path}: {str(e)}")
            raise

    def _classify_document(self, text: str) -> DocumentType:
        """Basic document classification"""
        text_lower = text.lower()

        if any(word in text_lower for word in ['lecture', 'slide', 'topic']):
            return DocumentType.SLIDES
        elif any(word in text_lower for word in ['textbook', 'chapter']):
            return DocumentType.TEXTBOOK
        elif any(word in text_lower for word in ['question', 'exam']):
            return DocumentType.QUESTION_PAPER
        elif any(word in text_lower for word in ['lab', 'experiment']):
            return DocumentType.LAB_MANUAL
        else:
            return DocumentType.LECTURE_NOTES

In [ ]:
class DocumentClassifier:
    """Advanced document classifier using NLP"""

    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.stop_words = set(nltk.corpus.stopwords.words('english'))
        logger.info("DocumentClassifier initialized with spaCy")

        # Keyword patterns for different document types
        self.patterns = {
            DocumentType.TEXTBOOK: [
                'chapter', 'section', 'exercise', 'review questions',
                'summary', 'key terms', 'bibliography', 'index',
                'appendix', 'glossary', 'figure', 'table'
            ],
            DocumentType.LECTURE_NOTES: [
                'lecture', 'notes', 'professor', 'class', 'today\'s topic',
                'recap', 'agenda', 'objectives', 'key points', 'discussion'
            ],
            DocumentType.QUESTION_PAPER: [
                                'question paper', 'marks', 'time allowed', 'section a',
                'section b', 'section c', 'attempt all questions',
                'maximum marks', 'instruction', 'answer any'
            ],
            DocumentType.LAB_MANUAL: [
                'lab', 'experiment', 'apparatus', 'procedure',
                'observation', 'result', 'conclusion', 'aim',
                'materials required', 'theory', 'precautions'
            ],
            DocumentType.SLIDES: [
                'slide', 'presentation', 'overview', 'agenda',
                'key points', 'summary', 'conclusion', 'thanks'
            ]
        }

    def classify(self, document: ProcessedDocument) -> DocumentType:
        """
        Classify document based on content and metadata
        """
        text = document.content.lower()
        scores = {}
        for doc_type, keywords in self.patterns.items():
            score = sum(1 for keyword in keywords if keyword in text)
            scores[doc_type] = score

        # Get document type with highest score
        if max(scores.values()) > 0:
            predicted_type = max(scores, key=scores.get)
        else:
            predicted_type = DocumentType.OTHER

        # Update document type
        document.document_type = predicted_type
        document.metadata['classification_scores'] = scores
        document.metadata['classification_method'] = 'nlp_keyword'

        return predicted_type

    def extract_topics(self, text: str) -> List[str]:
        """
        Extract main topics from document using NLP
         """
        doc = self.nlp(text[:10000])  # Process first 10000 chars for efficiency

        # Extract noun chunks as potential topics
        topics = []
        for chunk in doc.noun_chunks:
            # Filter out stop words and short phrases
            words = [token.text for token in chunk if token.text.lower() not in self.stop_words]
            if len(words) >= 2 and len(' '.join(words)) > 3:
                topics.append(' '.join(words))

        # Get unique topics (limited to top 10)
        seen = set()
        unique_topics = []
        for topic in topics:
            if topic not in seen and len(unique_topics) < 10:
                seen.add(topic)
                unique_topics.append(topic)

        return unique_topics


In [ ]:
class DocumentProcessingPipeline:
    """Main pipeline for processing all document types"""

    def __init__(self):
        self.pdf_loader = PDFLoader()
        self.docx_loader = DOCXLoader()
        self.pptx_loader = PPTXLoader()
        self.classifier = DocumentClassifier()

        self.processed_documents = []
        logger.info("DocumentProcessingPipeline initialized")

    def process_file(self, file_path: str) -> List[ProcessedDocument]:
        """
        Process a single file based on its extension
        """
        file_extension = Path(file_path).suffix.lower()

        if file_extension == '.pdf':
            documents = self.pdf_loader.load(file_path)
        elif file_extension == '.docx':
            documents = self.docx_loader.load(file_path)
        elif file_extension == '.pptx':
            documents = self.pptx_loader.load(file_path)
        else:
            raise ValueError(f"Unsupported file type: {file_extension}")

        # Classify each document
        for doc in documents:
            self.classifier.classify(doc)

            # Extract topics
            topics = self.classifier.extract_topics(doc.content)
            doc.metadata['topics'] = topics

        self.processed_documents.extend(documents)
        return documents

    def process_multiple_files(self, file_paths: List[str]) -> List[ProcessedDocument]:
        """
        Process multiple files
        """
        all_documents = []
        for file_path in file_paths:
            try:
                documents = self.process_file(file_path)
                all_documents.extend(documents)
                logger.info(f"Successfully processed: {file_path}")
            except Exception as e:
                logger.error(f"Failed to process {file_path}: {str(e)}")

        return all_documents

    def get_statistics(self) -> Dict[str, Any]:
        """
        Get processing statistics
        """
        stats = {
            'total_documents': len(self.processed_documents),
            'by_type': {},
            'by_source': {},
            'total_chars': 0
            }

        for doc in self.processed_documents:
            # Count by document type
            doc_type = doc.document_type.value
            stats['by_type'][doc_type] = stats['by_type'].get(doc_type, 0) + 1

            # Count by source
            source = doc.metadata.get('file_name', 'unknown')
            stats['by_source'][source] = stats['by_source'].get(source, 0) + 1

            # Count characters
            stats['total_chars'] += len(doc.content)

        return stats

In [ ]:
def create_sample_files():
    """Create sample files for testing"""

    # Sample PDF content (simulated)
    with open('/content/sample_lecture_notes.txt', 'w') as f:
        f.write("""Lecture Notes: Database Management Systems
Professor: Dr. Smith
Date: January 15, 2024

Topic: Introduction to Database Normalization

Normalization is the process of organizing data to reduce redundancy.
First Normal Form (1NF): Eliminate repeating groups
Second Normal Form (2NF): Remove partial dependencies
Third Normal Form (3NF): Remove transitive dependencies

Example:
Consider a student database table...
""")

    with open('/content/sample_question_paper.txt', 'w') as f:
        f.write("""Database Management Systems - End Semester Examination
Time: 3 hours
Maximum Marks: 100
Section A (20 marks)
Answer any 4 questions

Q1. Explain the concept of database normalization with examples.
Q2. What are ACID properties in transactions?
Q3. Differentiate between SQL and NoSQL databases.

Section B (40 marks)
Answer any 2 questions
...
""")

    logger.info("Sample files created for testing")

# Create sample files
create_sample_files()

# Initialize pipeline
pipeline = DocumentProcessingPipeline()
print("="*60)
print("TESTING DOCUMENT PROCESSING PIPELINE")
print("="*60)

# For demonstration, we'll use our sample text files
# In real scenario, these would be actual PDF/DOCX/PPTX files

# Simulate processing with text content
sample_docs = [
    ProcessedDocument(
        content="""Chapter 1: Introduction to Databases
A database is an organized collection of data. Database Management Systems (DBMS) are software systems used to store, retrieve, and manage data efficiently.

Key Concepts:
- Data independence
- Data integrity
- Data security
- Concurrent access

Types of Databases:
1. Relational Databases
2. NoSQL Databases
3. Object-oriented Databases""",
        metadata={'source': 'sample_textbook.txt', 'file_name': 'textbook_ch1.txt'},
        document_type=DocumentType.OTHER
    ),
    ProcessedDocument(
        content="""Lecture 5: Normalization
Date: 2024-01-20

Normalization Goals:
- Eliminate redundant data
- Ensure data dependencies make sense

Normal Forms:
1NF: Atomic values2NF: No partial dependencies
3NF: No transitive dependencies

Example:
Student(StudentID, Name, Course, Instructor)
Issues: Redundancy, Update anomalies""",
        metadata={'source': 'sample_notes.txt', 'file_name': 'lecture_5.txt'},
        document_type=DocumentType.OTHER
    ),
    ProcessedDocument(
        content="""Final Examination - Database Systems
Maximum Marks: 100
Time: 3 hours

Q1. What is normalization? Explain 1NF, 2NF, and 3NF with examples. (20 marks)
Q2. Consider the following relation and normalize it to 3NF... (25 marks)
Q3. Explain ACID properties with examples. (15 marks)""",
        metadata={'source': 'sample_exam.txt', 'file_name': 'exam_2023.txt'},
        document_type=DocumentType.OTHER
    )
]
classifier = DocumentClassifier()
for i, doc in enumerate(sample_docs):
    doc_type = classifier.classify(doc)
    topics = classifier.extract_topics(doc.content)
    doc.metadata['topics'] = topics

    print(f"\nDocument {i+1}:")
    print(f"  File: {doc.metadata['file_name']}")
    print(f"  Classified as: {doc_type.value}")
    print(f"  Content length: {len(doc.content)} chars")
    print(f"  Extracted topics: {topics[:5]}")
    pipeline.processed_documents = sample_docs

# Get statistics
stats = pipeline.get_statistics()
print("\n" + "="*60)
print("PROCESSING STATISTICS")
print("="*60)
print(f"Total documents processed: {stats['total_documents']}")
print(f"Total characters: {stats['total_chars']}")
print("\nDocuments by type:")
for doc_type, count in stats['by_type'].items():
    print(f"  {doc_type}: {count}")
print("\nDocuments by source:")
for source, count in stats['by_source'].items():
    print(f"  {source}: {count}")

/usr/local/lib/python3.12/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.4). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


TESTING DOCUMENT PROCESSING PIPELINE

Document 1:
  File: textbook_ch1.txt
  Classified as: textbook
  Content length: 374 chars
  Extracted topics: ['Databases \n database', 'organized collection', 'Database Management Systems', 'software systems', 'Key Concepts']

Document 2:
  File: lecture_5.txt
  Classified as: lecture_notes
  Content length: 314 chars
  Extracted topics: ['Lecture 5 : Normalization \n Date', '2024 - 01 - 20 \n\n Normalization Goals', 'redundant data \n - Ensure data dependencies', 'sense \n\n Normal Forms', 'partial dependencies']

Document 3:
  File: exam_2023.txt
  Classified as: question_paper
  Content length: 277 chars
  Extracted topics: ['Final Examination - Database Systems \n Maximum Marks', '100 \n Time', '3 hours \n\n Q1', '( 20 marks', 'following relation']

PROCESSING STATISTICS
Total documents processed: 3
Total characters: 965

Documents by type:
  textbook: 1
  lecture_notes: 1
  question_paper: 1

Documents by source:
  textbook_ch1.txt: 1
  lect

In [ ]:
import pickle

# Save processed documents to file
with open('/content/processed_docs_week1.pkl', 'wb') as f:
    pickle.dump(pipeline.processed_documents, f)

print("\n✅ Week 1 completed successfully!")
print("✅ Processed documents saved for Week 2")
print("\n📁 Files created:")
print("  - PDFLoader class")
print("  - DOCXLoader class")
print("  - PPTXLoader class")
print("  - DocumentClassifier class")
print("  - DocumentProcessingPipeline class")
print("  - processed_docs_week1.pkl")


✅ Week 1 completed successfully!
✅ Processed documents saved for Week 2

📁 Files created:
  - PDFLoader class
  - DOCXLoader class
  - PPTXLoader class
  - DocumentClassifier class
  - DocumentProcessingPipeline class
  - processed_docs_week1.pkl


In [ ]:
!pip install -q langchain==0.1.0 langchain-text-splitters==0.1.0
!pip install -q nltk==3.8.1 spacy==3.7.2
!pip install -q textstat==0.7.3
!python -m spacy download en_core_web_sm
!python -m spacy download en_core_web_lg

ERROR: Could not find a version that satisfies the requirement langchain-text-splitters==0.1.0 (from versions: 0.0.1, 0.0.2, 0.2.0, 0.2.1, 0.2.2, 0.2.4, 0.3.0.dev0, 0.3.0.dev1, 0.3.0, 0.3.1, 0.3.2, 0.3.3, 0.3.4, 0.3.5, 0.3.6rc1, 0.3.6rc2, 0.3.6, 0.3.7, 0.3.8, 0.3.9, 0.3.10, 0.3.11, 1.0.0a1, 1.0.0, 1.1.0, 1.1.1)
ERROR: No matching distribution found for langchain-text-splitters==0.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.1/105.1 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 17.7 MB/s eta 0:00:00
  ERROR: HTTP error 404 while getting https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz
ERROR: Could not install requirement https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz because of HTTP error 404 Client Error: Not Found for url: https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz for URL https://github.co

In [ ]:
import re
import pickle
import nltk
import spacy
from typing import List, Dict, Any, Tuple, Optional
from dataclasses import dataclass, field
from enum import Enum
import hashlib
import textstat
from collections import Counter
import numpy as np
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [ ]:
# ===============================
# Install and Import Libraries
# ===============================

import os
import pickle
import logging
from dataclasses import dataclass
from typing import Dict

import nltk
import spacy

# Download required NLTK data
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


# ===============================
# Define Data Class
# ===============================

@dataclass
class ProcessedDocument:
    content: str
    metadata: Dict
    document_type: str


# ===============================
# Load Week 1 Processed Documents
# ===============================

try:
    with open('/content/processed_docs_week1.pkl', 'rb') as f:
        processed_docs = pickle.load(f)

    print(f"✅ Loaded {len(processed_docs)} processed documents from Week 1")

except:
    print("⚠️ No processed documents found. Creating sample documents...")

    processed_docs = [

        ProcessedDocument(
            content="""
Database normalization is the process of structuring a relational database in accordance with a series of normal forms to reduce data redundancy and improve data integrity.

Normal Forms:

First Normal Form (1NF):
A relation is in 1NF if it contains only atomic values and no repeating groups.

Second Normal Form (2NF):
A relation is in 2NF if it is in 1NF and all non-key attributes depend on the entire primary key.

Third Normal Form (3NF):
A relation is in 3NF if it is in 2NF and there are no transitive dependencies.

Boyce-Codd Normal Form (BCNF):
A stronger version of 3NF where every determinant is a candidate key.

Example:
Student(StudentID, Name, Course, Instructor)

Solution:
Split into
Student(StudentID, Name)
Enrollment(StudentID, Course, Instructor)
""",

            metadata={
                'source': 'textbook_ch1.txt',
                'file_name': 'textbook_ch1.txt',
                'document_type': 'textbook'
            },

            document_type='textbook'
        ),


        ProcessedDocument(
            content="""
Lecture Notes: Normalization (Lecture 5)

Objectives
- Understand redundancy problems
- Learn normalization process
- Apply normal forms

1NF Rules
- Atomic values only
- No repeating groups
- Each row unique

2NF Requirements
- Must be in 1NF
- No partial dependencies

3NF Requirements
- Must be in 2NF
- No transitive dependencies

Practice:
Given R(A,B,C,D) with FD: A→B, B→C, C→D
Normalize to 3NF
""",

            metadata={
                'source': 'lecture_notes.txt',
                'file_name': 'lecture_5.txt',
                'document_type': 'lecture_notes'
            },

            document_type='lecture_notes'
        ),


        ProcessedDocument(
            content="""
Database Management Systems - End Term Exam 2023

Section A

Q1. What is normalization?
Q2. Define 1NF with example.
Q3. What is functional dependency?
Q4. Explain partial dependency.

Section B

Q5. Normalize relation to 3NF:
R(StudentID, StudentName, CourseID, CourseName, Instructor, Grade)

FDs:
StudentID → StudentName
CourseID → CourseName
(StudentID, CourseID) → Grade

Q6. Difference between 2NF and 3NF.

Q7. Advantages of normalization.
""",

            metadata={
                'source': 'exam_2023.txt',
                'file_name': 'exam_2023.txt',
                'document_type': 'question_paper'
            },

            document_type='question_paper'
        )
    ]

    print(f"✅ Created {len(processed_docs)} sample documents")


# ===============================
# Save Documents for Later Weeks
# ===============================

with open('/content/processed_docs_week1.pkl', 'wb') as f:
    pickle.dump(processed_docs, f)

print("💾 Documents saved successfully for next weeks.")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
/usr/local/lib/python3.12/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


⚠️ No processed documents found. Creating sample documents...
✅ Created 3 sample documents
💾 Documents saved successfully for next weeks.


In [ ]:
class TextCleaner:
    """Advanced text cleaning and preprocessing"""

    def __init__(self):
        self.stop_words = set(stopwords.words('english'))
        self.lemmatizer = WordNetLemmatizer()

    def clean_text(self, text: str, options: Dict[str, bool] = None) -> str:
        """
        Clean text based on specified options
        """
        if options is None:
            options = {
                'remove_special_chars': True,
                'remove_extra_spaces': True,
                'remove_numbers': False,
                'lowercase': True,
                'remove_stopwords': False,
                'lemmatize': False
            }
            original = text

        # Remove special characters
        if options.get('remove_special_chars', True):
            text = re.sub(r'[^\w\s\.\,\;\:\?\-\!]', ' ', text)

        # Remove extra spaces
        if options.get('remove_extra_spaces', True):
            text = re.sub(r'\s+', ' ', text)

        # Remove numbers
        if options.get('remove_numbers', False):
            text = re.sub(r'\d+', '', text)
            if options.get('lowercase', True):
               text = text.lower()

        # Remove stopwords
        if options.get('remove_stopwords', False):
            words = text.split()
            words = [w for w in words if w not in self.stop_words]
            text = ' '.join(words)

        # Lemmatize
        if options.get('lemmatize', False):
            words = text.split()
            words = [self.lemmatizer.lemmatize(w) for w in words]
            text = ' '.join(words)

        # Strip whitespace
        text = text.strip()

        return text
        def extract_sentences(self, text: str) -> List[str]:
            """Extract sentences from text"""
            return sent_tokenize(text)

    def extract_paragraphs(self, text: str) -> List[str]:
        """Extract paragraphs from text"""
        # Split by double newline or common paragraph markers
        paragraphs = re.split(r'\n\s*\n', text)
        return [p.strip() for p in paragraphs if p.strip()]

    def get_readability_score(self, text: str) -> Dict[str, float]:
        """Calculate readability metrics"""
        return {
            'flesch_reading_ease': textstat.flesch_reading_ease(text),
            'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
            'gunning_fog': textstat.gunning_fog(text),
            'smog_index': textstat.smog_index(text),
            'coleman_liau_index': textstat.coleman_liau_index(text),
            'automated_readability_index': textstat.automated_readability_index(text)
        }

In [ ]:
# CELL 4: Intelligent Chunking Strategies
class ChunkingStrategy(Enum):
    """Available chunking strategies"""
    FIXED_SIZE = "fixed_size"
    PARAGRAPH = "paragraph"
    SEMANTIC = "semantic"
    SENTENCE = "sentence"
    TOPIC_BASED = "topic_based"

@dataclass
class TextChunk:
    """Data class for text chunks"""
    text: str
    chunk_id: str
    metadata: Dict[str, Any]
    embeddings: Optional[np.ndarray] = None
    start_idx: int = 0
    end_idx: int = 0

class IntelligentChunker:
    """Advanced text chunking with multiple strategies"""

    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.cleaner = TextCleaner()

    def chunk_by_fixed_size(self, text: str, chunk_size: int = 500,
                           overlap: int = 50) -> List[TextChunk]:
        """Chunk by fixed token count with overlap"""
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk_words = words[i:i + chunk_size]
            if not chunk_words:
                break

            chunk_text = ' '.join(chunk_words)

            # Create chunk metadata
            chunk = TextChunk(
                text=chunk_text,
                chunk_id=hashlib.md5(f"{i}{chunk_text[:50]}".encode()).hexdigest()[:8],
                metadata={
                    'strategy': 'fixed_size',
                    'chunk_size': chunk_size,
                    'overlap': overlap,
                    'start_word_idx': i,
                    'end_word_idx': i + len(chunk_words),
                    'word_count': len(chunk_words)
                },
                start_idx=i,
                end_idx=i + len(chunk_words)
            )
            chunks.append(chunk)

        return chunks

    def chunk_by_paragraph(self, text: str) -> List[TextChunk]:
        """Chunk by paragraphs"""
        paragraphs = self.cleaner.extract_paragraphs(text)
        chunks = []

        for i, para in enumerate(paragraphs):
            if len(para.split()) < 10:  # Skip very short paragraphs
                continue

            chunk = TextChunk(
                text=para,
                chunk_id=hashlib.md5(f"para_{i}_{para[:50]}".encode()).hexdigest()[:8],
                metadata={
                    'strategy': 'paragraph',
                    'paragraph_index': i,
                    'word_count': len(para.split()),
                    'char_count': len(para)
                },
                start_idx=i,
                end_idx=i
            )
            chunks.append(chunk)

        return chunks

    def chunk_by_sentence(self, text: str, max_sentences: int = 5) -> List[TextChunk]:
        """Chunk by groups of sentences"""
        sentences = self.cleaner.extract_sentences(text)
        chunks = []

        for i in range(0, len(sentences), max_sentences):
            chunk_sentences = sentences[i:i + max_sentences]
            chunk_text = ' '.join(chunk_sentences)

            chunk = TextChunk(
                text=chunk_text,
                chunk_id=hashlib.md5(f"sent_{i}_{chunk_text[:50]}".encode()).hexdigest()[:8],
                metadata={
                    'strategy': 'sentence',
                    'start_sentence': i,
                    'end_sentence': i + len(chunk_sentences),
                    'sentence_count': len(chunk_sentences),
                    'word_count': len(chunk_text.split())
                },
                start_idx=i,
                end_idx=i + len(chunk_sentences)
            )
            chunks.append(chunk)

        return chunks

    def chunk_semantic(self, text: str, similarity_threshold: float = 0.7) -> List[TextChunk]:
        """Chunk based on semantic similarity using spaCy"""
        doc = self.nlp(text)
        sentences = list(doc.sents)
        chunks = []

        current_chunk = []
        current_chunk_text = ""

        for i, sent in enumerate(sentences):
            sent_text = sent.text.strip()

            if not current_chunk:
                current_chunk.append(sent_text)
                current_chunk_text = sent_text
            else:
                # Check semantic similarity with last sentence in chunk
                last_sent = current_chunk[-1]
                last_doc = self.nlp(last_sent)
                current_doc = self.nlp(sent_text)

                if last_doc.vector is not None and current_doc.vector is not None:
                    similarity = np.dot(last_doc.vector, current_doc.vector) / (
                        np.linalg.norm(last_doc.vector) * np.linalg.norm(current_doc.vector)
                    )

                    if similarity > similarity_threshold:
                        current_chunk.append(sent_text)
                        current_chunk_text += " " + sent_text
                    else:
                        # Save current chunk
                        if current_chunk:
                            chunk = TextChunk(
                                text=current_chunk_text,
                                chunk_id=hashlib.md5(f"sem_{i}_{current_chunk_text[:50]}".encode()).hexdigest()[:8],
                                metadata={
                                    'strategy': 'semantic',
                                    'sentence_count': len(current_chunk),
                                    'avg_similarity': similarity
                                }
                            )
                            chunks.append(chunk)

                        # Start new chunk
                        current_chunk = [sent_text]
                        current_chunk_text = sent_text

        # Add last chunk
        if current_chunk:
            chunk = TextChunk(
                text=current_chunk_text,
                chunk_id=hashlib.md5(f"sem_last_{current_chunk_text[:50]}".encode()).hexdigest()[:8],
                metadata={
                    'strategy': 'semantic',
                    'sentence_count': len(current_chunk)
                }
            )
            chunks.append(chunk)

        return chunks

    def chunk_by_topic(self, text: str, num_topics: int = 5) -> List[TextChunk]:
        """Chunk based on topic boundaries"""
        # This is a simplified version - in production, use BERTopic or LDA
        paragraphs = self.cleaner.extract_paragraphs(text)
        chunks = []

        # Simple topic detection using noun phrases
        topics_detected = []
        for para in paragraphs:
            doc = self.nlp(para[:500])  # First 500 chars
            noun_phrases = [chunk.text for chunk in doc.noun_chunks if len(chunk.text.split()) <= 3]
            if noun_phrases:
                topics_detected.append(noun_phrases[0])  # Use first noun phrase as topic

        # Group paragraphs with similar topics
        current_topic = None
        current_chunk = []
        current_chunk_text = ""

        for i, para in enumerate(paragraphs):
            para_topic = topics_detected[i] if i < len(topics_detected) else None

            if current_topic is None or para_topic == current_topic:
                current_topic = para_topic
                current_chunk.append(para)
                current_chunk_text += para + "\n\n"
            else:
                # Save current chunk
                if current_chunk:
                    chunk = TextChunk(
                        text=current_chunk_text.strip(),
                        chunk_id=hashlib.md5(f"topic_{i}_{current_chunk_text[:50]}".encode()).hexdigest()[:8],
                        metadata={
                            'strategy': 'topic_based',
                            'topic': current_topic,
                            'paragraph_count': len(current_chunk)
                        }
                    )
                    chunks.append(chunk)

                # Start new chunk
                current_topic = para_topic
                current_chunk = [para]
                current_chunk_text = para + "\n\n"

        # Add last chunk
        if current_chunk:
            chunk = TextChunk(
                text=current_chunk_text.strip(),
                chunk_id=hashlib.md5(f"topic_last_{current_chunk_text[:50]}".encode()).hexdigest()[:8],
                metadata={
                    'strategy': 'topic_based',
                    'topic': current_topic,
                    'paragraph_count': len(current_chunk)
                }
            )
            chunks.append(chunk)

        return chunks



In [ ]:
# CELL 5: Metadata Extractor
class MetadataExtractor:
    """Extract rich metadata from text chunks"""

    def __init__(self):
        self.nlp = spacy.load("en_core_web_lg")
        self.stop_words = set(stopwords.words('english'))

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        """Extract named entities from text"""
        doc = self.nlp(text[:10000])  # Limit for performance

        entities = {
            'PERSON': [],
            'ORG': [],
            'GPE': [],
            'DATE': [],
            'MONEY': [],
            'PERCENT': [],
            'PRODUCT': [],
            'EVENT': [],
            'WORK_OF_ART': [],
            'LAW': [],
            'LANGUAGE': []
        }

        for ent in doc.ents:
            if ent.label_ in entities:
                if ent.text not in entities[ent.label_]:
                    entities[ent.label_].append(ent.text)

        return {k: v for k, v in entities.items() if v}

    def extract_key_phrases(self, text: str, top_n: int = 10) -> List[str]:
        """Extract key phrases using noun chunks"""
        doc = self.nlp(text[:10000])

        phrases = []
        for chunk in doc.noun_chunks:
            # Filter out stop words and short phrases
            words = [token.text for token in chunk if token.text.lower() not in self.stop_words]
            if len(words) >= 2 and len(' '.join(words)) > 3:
                phrases.append(' '.join(words))

        # Count frequency
        phrase_freq = Counter(phrases)

        # Return top N most common phrases
        return [phrase for phrase, _ in phrase_freq.most_common(top_n)]

    def extract_technical_terms(self, text: str) -> List[str]:
        """Extract technical/specialized terms"""
        doc = self.nlp(text[:10000])

        technical_terms = []

        for token in doc:
            # Look for capitalized terms, acronyms, or terms with special patterns
            if (token.text.isupper() and len(token.text) >= 2) or \
               (token.text[0].isupper() and token.pos_ == 'PROPN') or \
               (token.pos_ == 'NOUN' and len(token.text) > 5):
                if token.text.lower() not in self.stop_words:
                    technical_terms.append(token.text)

        return list(set(technical_terms))[:20]

    def extract_questions(self, text: str) -> List[str]:
        """Extract questions from text"""
        # Look for sentences ending with ?
        questions = re.findall(r'[^.!?]*\?', text)
        return [q.strip() for q in questions if q.strip()]

    def extract_code_snippets(self, text: str) -> List[str]:
        """Extract potential code snippets"""
        # Look for code-like patterns
        code_patterns = [
            r'```.*?```',  # Markdown code blocks
            r'(?:def|class|import|if|for|while|return)\s+\w+',  # Python-like
            r'CREATE\s+TABLE|SELECT\s+.*\s+FROM|INSERT\s+INTO',  # SQL
        ]

        snippets = []
        for pattern in code_patterns:
            matches = re.findall(pattern, text, re.DOTALL | re.IGNORECASE)
            snippets.extend(matches)

        return snippets

In [ ]:
# CELL 6: Complete Processing Pipeline
class TextProcessingPipeline:
    """Complete text processing pipeline"""

    def __init__(self):
        self.cleaner = TextCleaner()
        self.chunker = IntelligentChunker()
        self.metadata_extractor = MetadataExtractor()
        self.processed_chunks = []

    def process_document(self, doc, chunk_strategy: ChunkingStrategy = ChunkingStrategy.PARAGRAPH,
                         chunk_params: Dict = None) -> List[TextChunk]:
        """
        Process a single document through the entire pipeline
        """
        if chunk_params is None:
            chunk_params = {}

        # Step 1: Clean text
        cleaned_text = self.cleaner.clean_text(
            doc.content,
            options={'remove_special_chars': True, 'remove_extra_spaces': True, 'lowercase': False}
        )

        # Step 2: Choose chunking strategy
        if chunk_strategy == ChunkingStrategy.FIXED_SIZE:
            chunks = self.chunker.chunk_by_fixed_size(
                cleaned_text,
                chunk_size=chunk_params.get('chunk_size', 500),
                overlap=chunk_params.get('overlap', 50)
            )
        elif chunk_strategy == ChunkingStrategy.PARAGRAPH:
            chunks = self.chunker.chunk_by_paragraph(cleaned_text)
        elif chunk_strategy == ChunkingStrategy.SENTENCE:
            chunks = self.chunker.chunk_by_sentence(
                cleaned_text,
                max_sentences=chunk_params.get('max_sentences', 5)
            )
        elif chunk_strategy == ChunkingStrategy.SEMANTIC:
            chunks = self.chunker.chunk_semantic(
                cleaned_text,
                similarity_threshold=chunk_params.get('similarity_threshold', 0.7)
            )
        elif chunk_strategy == ChunkingStrategy.TOPIC_BASED:
            chunks = self.chunker.chunk_by_topic(
                cleaned_text,
                num_topics=chunk_params.get('num_topics', 5)
            )
        else:
            chunks = self.chunker.chunk_by_paragraph(cleaned_text)

        # Step 3: Extract metadata for each chunk
        for i, chunk in enumerate(chunks):
            # Add document metadata
            chunk.metadata.update({
                'doc_source': doc.metadata.get('source', 'unknown'),
                'doc_type': doc.document_type,
                'doc_file': doc.metadata.get('file_name', 'unknown'),
                'chunk_index': i,
                'total_chunks': len(chunks)
            })

            # Extract rich metadata
            chunk.metadata['entities'] = self.metadata_extractor.extract_entities(chunk.text)
            chunk.metadata['key_phrases'] = self.metadata_extractor.extract_key_phrases(chunk.text, top_n=5)
            chunk.metadata['technical_terms'] = self.metadata_extractor.extract_technical_terms(chunk.text)

            questions = self.metadata_extractor.extract_questions(chunk.text)
            if questions:
                chunk.metadata['questions'] = questions

            # Calculate readability
            readability = self.cleaner.get_readability_score(chunk.text)
            chunk.metadata['readability'] = readability

        self.processed_chunks.extend(chunks)
        return chunks

    def process_all_documents(self, documents: List, chunk_strategy: ChunkingStrategy = ChunkingStrategy.PARAGRAPH):
        """Process all documents"""
        all_chunks = []
        for doc in documents:
            chunks = self.process_document(doc, chunk_strategy)
            all_chunks.extend(chunks)
            print(f"✅ Processed document: {doc.metadata.get('file_name', 'unknown')} -> {len(chunks)} chunks")

        return all_chunks

    def get_chunk_statistics(self) -> Dict[str, Any]:
        """Get statistics about processed chunks"""
        stats = {
            'total_chunks': len(self.processed_chunks),
            'avg_chunk_length': 0,
            'chunk_length_distribution': {},
            'by_doc_type': {},
            'total_entities': 0,
            'total_key_phrases': 0
        }

        if not self.processed_chunks:
            return stats

        lengths = [len(chunk.text.split()) for chunk in self.processed_chunks]
        stats['avg_chunk_length'] = np.mean(lengths)

        # Distribution
        bins = [0, 50, 100, 200, 500, 1000]
        labels = ['0-50', '51-100', '101-200', '201-500', '501-1000']
        for length in lengths:
            for i, bin_edge in enumerate(bins[1:]):
                if length <= bin_edge:
                    label = labels[i]
                    stats['chunk_length_distribution'][label] = stats['chunk_length_distribution'].get(label, 0) + 1
                    break

        # By document type
        for chunk in self.processed_chunks:
            doc_type = chunk.metadata.get('doc_type', 'unknown')
            stats['by_doc_type'][doc_type] = stats['by_doc_type'].get(doc_type, 0) + 1

            # Count metadata
            stats['total_entities'] += len(chunk.metadata.get('entities', {}))
            stats['total_key_phrases'] += len(chunk.metadata.get('key_phrases', []))

        return stats

In [ ]:
# Install compatible versions
!pip install -q numpy==1.26.4
!pip install -q spacy==3.7.2
!pip install -q nltk==3.8.1
!pip install -q PyPDF2 python-docx python-pptx

# Download spaCy model
!python -m spacy download en_core_web_sm

  ERROR: HTTP error 404 while getting https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz
ERROR: Could not install requirement https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz because of HTTP error 404 Client Error: Not Found for url: https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz for URL https://github.com/explosion/spacy-models/releases/download/-en_core_web_sm/-en_core_web_sm.tar.gz


In [ ]:
import os
import logging
from pathlib import Path
from typing import List, Dict, Any
from dataclasses import dataclass
from enum import Enum
import hashlib
import json
from datetime import datetime

import PyPDF2
from docx import Document
from pptx import Presentation

import spacy
import nltk

# Download NLTK data
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
nltk.download('stopwords')

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
# SAFE MODEL LOADING (prevents crash)

try:
    nlp = spacy.load("en_core_web_sm")
except:
    import os
    os.system("python -m spacy download en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

/usr/local/lib/python3.12/dist-packages/spacy/util.py:910: UserWarning: [W095] Model 'en_core_web_sm' (3.8.0) was trained with spaCy v3.8.0 and may not be 100% compatible with the current version (3.7.2). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)


In [ ]:
class TextProcessingPipeline:

    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.chunks = []

    def process_all_documents(self, docs, strategy):
        results = []

        for doc in docs:
            text = doc["text"]

            # simple paragraph chunking
            paragraphs = text.split("\n")

            for p in paragraphs:
                if len(p.strip()) > 20:
                    results.append({
                        "text": p,
                        "metadata": {
                            "doc_type": doc.get("type", "unknown")
                        }
                    })

        self.chunks = results
        return results

    def get_chunk_statistics(self):

        total = len(self.chunks)

        avg_len = sum(len(c["text"].split()) for c in self.chunks)/total if total else 0

        return {
            "total_chunks": total,
            "avg_chunk_length": avg_len,
            "by_doc_type": {}
        }

In [ ]:
# CREATE processed_docs VARIABLE (REQUIRED)

processed_docs = []

# Example test document (replace later with your real files)

sample_text = """
Artificial Intelligence is transforming education.
AI based subject guide systems help students find answers quickly.
Question bank AI agents generate questions and help students practice.
"""

processed_docs.append({
    "text": sample_text,
    "type": "text",
    "source": "sample_document"
})

print("Documents loaded:", len(processed_docs))

Documents loaded: 1


In [ ]:
pipeline = TextProcessingPipeline()

chunks = pipeline.process_all_documents(processed_docs, "paragraph")

stats = pipeline.get_chunk_statistics()

print("Total chunks:", stats["total_chunks"])
print("Average length:", stats["avg_chunk_length"])

Total chunks: 3
Average length: 8.333333333333334


In [ ]:
for i, chunk in enumerate(chunks):
    print("\nChunk", i+1)
    print(chunk["text"])


Chunk 1
Artificial Intelligence is transforming education.

Chunk 2
AI based subject guide systems help students find answers quickly.

Chunk 3
Question bank AI agents generate questions and help students practice.


In [ ]:
# Import required library
import pickle

# STEP 1 — Create sample processed_docs if not already defined
try:
    processed_docs
except NameError:
    print("processed_docs not found. Creating sample documents...")

    processed_docs = [
        {
            "text": """Database normalization is the process of organizing data in a database
            to reduce redundancy and improve integrity. First Normal Form (1NF) ensures
            atomic values. Second Normal Form (2NF) removes partial dependencies.
            Third Normal Form (3NF) removes transitive dependencies.""",
            "metadata": {"doc_type": "textbook"}
        }
    ]


# STEP 2 — Safe Chunking Function
def create_chunks(docs, chunk_size=100):

    chunks = []

    for doc in docs:

        # safely get text
        text = doc.get("text", "")

        # safely get metadata
        metadata = doc.get("metadata", {"doc_type": "unknown"})

        words = text.split()

        for i in range(0, len(words), chunk_size):

            chunk_text = " ".join(words[i:i+chunk_size])

            chunks.append({
                "text": chunk_text,
                "metadata": metadata
            })

    return chunks


# STEP 3 — Generate chunks
all_chunks = create_chunks(processed_docs)

print("Chunks created:", len(all_chunks))


# STEP 4 — Save Week 2 chunks
with open('/content/processed_chunks_week2.pkl', 'wb') as f:
    pickle.dump(all_chunks, f)


print("\n✅ Week 2 completed successfully!")
print("✅ processed_chunks_week2.pkl created")
print("✅ Total chunks created:", len(all_chunks))


Chunks created: 1

✅ Week 2 completed successfully!
✅ processed_chunks_week2.pkl created
✅ Total chunks created: 1


In [ ]:
!pip install -q sentence-transformers==2.2.2
!pip install -q faiss-cpu==1.7.4
!pip install -q numpy pandas
!pip install -q langchain==0.1.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires typer<1.0,>=0.12, but you have typer 0.9.4 which is incompatible.
ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2)
ERROR: No matching distribution found for faiss-cpu==1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.0/798.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Fix sentence-transformers compatibility
!pip uninstall -y huggingface_hub
!pip install huggingface_hub==0.16.4
!pip install sentence-transformers==2.2.2
!pip install faiss-cpu

print("✅ Libraries fixed. Restart runtime after this cell.")

Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 3.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 4.57.6 requires huggingface-hub<1.0,>=0.34.0, but you have huggingface-hub 0.16.4 which is incompatible.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.16.4 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.16.4 which is incompatible.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.16.4 which is incompatible.
gradio 5.50.0 requires typer<1.0,>=0.12, but you have typer 0.9.4 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have h

In [ ]:
# Fix dependency conflict
!pip uninstall -y huggingface_hub sentence-transformers transformers

!pip install huggingface_hub==0.16.4
!pip install transformers==4.33.2
!pip install sentence-transformers==2.2.2
!pip install faiss-cpu

print("✅ Dependencies fixed successfully")

Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: sentence-transformers 2.2.2
Uninstalling sentence-transformers-2.2.2:
  Successfully uninstalled sentence-transformers-2.2.2
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
  Using cached huggingface_hub-0.16.4-py3-none-any.whl.metadata (12 kB)
Using cached huggingface_hub-0.16.4-py3-none-any.whl (268 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.16.4 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.16.4 which is

ERROR: Operation cancelled by user
^C
✅ Dependencies fixed successfully


In [ ]:
# Remove conflicting libraries
!pip uninstall -y huggingface_hub sentence-transformers transformers

# Install compatible versions
!pip install huggingface_hub==0.16.4
!pip install transformers==4.33.2
!pip install sentence-transformers==2.2.2
!pip install faiss-cpu

print("✅ Dependencies installed successfully")

Found existing installation: huggingface_hub 0.36.2
Uninstalling huggingface_hub-0.36.2:
  Successfully uninstalled huggingface_hub-0.36.2
Found existing installation: sentence-transformers 2.2.2
Uninstalling sentence-transformers-2.2.2:
  Successfully uninstalled sentence-transformers-2.2.2
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
  Using cached huggingface_hub-0.16.4-py3-none-any.whl.metadata (12 kB)
Using cached huggingface_hub-0.16.4-py3-none-any.whl (268 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.1 requires transformers, which is not installed.
peft 0.18.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.16.4 which is incompatible.
gradio-client 1.14.0 requires huggingface-hub<2.0,>=0.19.3, but you have huggingface-hub 0.16.4 which is

In [ ]:
!pip install transformers torch faiss-cpu

In [ ]:
import pickle
import numpy as np
import faiss
import torch
from transformers import AutoTokenizer, AutoModel
from typing import List, Dict
from dataclasses import dataclass
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("✅ Libraries imported successfully")


# Load embedding model
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

print("✅ Embedding model loaded")


# Function to generate embeddings
def get_embedding(text):

    inputs = tokenizer(
        text,
        padding=True,
        truncation=True,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**inputs)

    # Mean pooling
    embeddings = outputs.last_hidden_state.mean(dim=1)

    return embeddings.numpy()


# Example
text = "Database normalization reduces redundancy in databases."

embedding = get_embedding(text)

print("Embedding shape:", embedding.shape)

✅ Libraries imported successfully


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

✅ Embedding model loaded
Embedding shape: (1, 384)


In [ ]:
class EmbeddingGenerator:
    """Generate embeddings for text chunks using sentence-transformers"""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        """
        Initialize embedding model
        """
        self.model_name = model_name
        logger.info(f"Loading embedding model: {model_name}")

        self.model = SentenceTransformer(model_name)
        self.embedding_dim = self.model.get_sentence_embedding_dimension()

        logger.info(f"Model loaded. Embedding dimension: {self.embedding_dim}")

    def generate_embedding(self, text: str) -> np.ndarray:
        """
        Generate embedding for a single text
        """
        if len(text) > 10000:
            text = text[:10000]

        embedding = self.model.encode(text, normalize_embeddings=True)
        return embedding

    def generate_embeddings_batch(self, texts: List[str], batch_size: int = 32) -> np.ndarray:
        """
        Generate embeddings for multiple texts in batch
        """

        texts = [t[:10000] if len(t) > 10000 else t for t in texts]

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            normalize_embeddings=True,
            show_progress_bar=True
        )

        return embeddings

    def generate_for_chunks(self, chunks: List, batch_size: int = 32) -> List:
        """
        Generate embeddings for a list of chunks and attach them
        """

        texts = [chunk.text for chunk in chunks]

        logger.info(f"Generating embeddings for {len(texts)} chunks...")
        start_time = time.time()

        embeddings = self.generate_embeddings_batch(texts, batch_size)

        # Attach embeddings to chunks
        for chunk, embedding in zip(chunks, embeddings):
            chunk.embedding = embedding

        elapsed = time.time() - start_time
        logger.info(f"Embeddings generated in {elapsed:.2f} seconds")

        return chunks

In [ ]:
import pickle
import numpy as np
import faiss
from typing import List, Dict, Tuple, Any
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [ ]:
@dataclass
class TextChunk:
    text: str
    chunk_id: str
    metadata: Dict
    embedding: np.ndarray = None

In [ ]:
try:
    with open('/content/processed_chunks_week2.pkl', 'rb') as f:
        chunks = pickle.load(f)

    print("✅ Loaded Week2 chunks:", len(chunks))

except:
    print("⚠️ Creating sample chunks")

    chunks = [
        TextChunk(
            text="Database normalization reduces redundancy and improves integrity.",
            chunk_id="chunk1",
            metadata={"doc_type": "textbook"}
        ),
        TextChunk(
            text="First Normal Form requires atomic attributes.",
            chunk_id="chunk2",
            metadata={"doc_type": "lecture_notes"}
        ),
        TextChunk(
            text="Second Normal Form removes partial dependency.",
            chunk_id="chunk3",
            metadata={"doc_type": "lecture_notes"}
        ),
        TextChunk(
            text="Third Normal Form removes transitive dependency.",
            chunk_id="chunk4",
            metadata={"doc_type": "textbook"}
        )
    ]

✅ Loaded Week2 chunks: 1


In [ ]:
class EmbeddingGenerator:

    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):

        print("Loading embedding model...")
        self.model = SentenceTransformer(model_name)

        self.embedding_dim = self.model.get_sentence_embedding_dimension()

        print("Embedding dimension:", self.embedding_dim)

    def generate_embedding(self, text):

        embedding = self.model.encode(text, normalize_embeddings=True)

        return embedding

    def generate_for_chunks(self, chunks):

        texts = [c.text for c in chunks]

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True,
            normalize_embeddings=True
        )

        for chunk, emb in zip(chunks, embeddings):
            chunk.embedding = emb

        return chunks

In [ ]:
class FAISSVectorStore:

    def __init__(self, embedding_dim):

        self.index = faiss.IndexFlatL2(embedding_dim)

        self.id_to_chunk = {}

        self.next_id = 0

    def add_chunks(self, chunks):

        embeddings = []

        valid_chunks = []

        for c in chunks:
            if c.embedding is not None:
                embeddings.append(c.embedding)
                valid_chunks.append(c)

        embeddings = np.array(embeddings).astype("float32")

        self.index.add(embeddings)

        for i, chunk in enumerate(valid_chunks):
            self.id_to_chunk[self.next_id + i] = chunk

        self.next_id += len(valid_chunks)

        print("Added chunks:", len(valid_chunks))

    def search(self, query_embedding, k=3):

        query_embedding = np.array([query_embedding]).astype("float32")

        distances, indices = self.index.search(query_embedding, k)

        results = []

        for dist, idx in zip(distances[0], indices[0]):

            if idx in self.id_to_chunk:

                chunk = self.id_to_chunk[idx]

                similarity = 1 / (1 + dist)

                results.append((chunk, similarity))

        return results